In [ ]:
!mkdir ../checkpoints
!wget https://download.openmmlab.com/mmsegmentation/v0.5/pspnet/pspnet_r50-d8_512x1024_40k_cityscapes/pspnet_r50-d8_512x1024_40k_cityscapes_20200605_003338-2966598c.pth -P ../checkpoints

In [ ]:
import torch
import matplotlib.pyplot as plt
from mmengine.model.utils import revert_sync_batchnorm
from mmseg.apis import init_model, inference_model, show_result_pyplot

/home/cz/anaconda3/envs/mm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/cz/anaconda3/envs/mm/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/cz/anaconda3/envs/mm/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/cz/anaconda3/envs/mm/lib/python3.10/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


/home/cz/codes/segsRoad/vim


/home/cz/codes/segsRoad/mmseg/models/backbones/localvmamba.py:29: UserWarning: No module named 'selective_scan_cuda_core'
"selective_scan_cuda_core" not found, use default "selective_scan_cuda" instead.
  warnings.warn(f"{e}\n\"selective_scan_cuda_core\" not found, use default \"selective_scan_cuda\" instead.")


In [2]:
config_file = '../configs/fcn/deep_connect_fcn_r50-d8_4xb2-40k_cityscapes-512x1024.py'
checkpoint_file = '../work_dirs/deep_connect_fcn_r50-d8_4xb2-40k_cityscapes-512x1024/iter_40000.pth'

In [3]:
# build the model from a config file and a checkpoint file
model = init_model(config_file, checkpoint_file, device='cpu')

/home/cz/codes/segsRoad/mmseg/models/decode_heads/decode_head.py:120: UserWarning: For binary segmentation, we suggest using`out_channels = 1` to define the outputchannels of segmentor, and use `threshold`to convert `seg_logits` into a predictionapplying a threshold
  warnings.warn('For binary segmentation, we suggest using'
/home/cz/codes/segsRoad/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: ../work_dirs/deep_connect_fcn_r50-d8_4xb2-40k_cityscapes-512x1024/iter_40000.pth


In [ ]:
output = model.predict(torch.randn(1,3,512,512))

In [ ]:
output[0].get('seg_logits').data

torch.Size([2, 512, 512])

In [19]:
# test a single image
img = '/data1/datasets/zhengbo/roaddataset/deepglobe/images/test/744973_0_512.jpg'
if not torch.cuda.is_available():
    model = revert_sync_batchnorm(model)
result = inference_model(model, img)

In [20]:
output = result.get('seg_logits').data.softmax(dim=0)[1]
output.shape

torch.Size([512, 512])

In [24]:
output.unique()

tensor([3.5695e-19, 8.0408e-19, 9.0458e-19,  ..., 9.9164e-01, 9.9177e-01,
        9.9212e-01])

In [29]:
from PIL import Image
import numpy as np

Image.fromarray(np.array(output.cpu().numpy()*256,dtype=np.uint8)).save('logit_prob.png')

In [ ]:
# show the results
vis_result = show_result_pyplot(model, img, result, show=False)
plt.imshow(vis_result)